# B.O.S.S. - Test PicoDet-S 320 (ONNX) su recordings

**Famiglia:** PicoDet (anchor-free, Generalized Focal Loss) - diversa da YOLO ed EfficientDet.

**Pipeline:**
1. Ricerca frame (dataset gia estratto su Kaggle)
2. Caricamento modello ONNX + inferenza (preprocessing **letterbox**) -> frame annotati
3. Distribuzione classi e confidenza
4. Caricamento Ground Truth (export Roboflow **COCO JSON**, stesso split canonico di EfficientDet)
5. Valutazione quantitativa contro GT (mAP, Precision/Recall reali via TP/FP/FN)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo
8. Report Markdown (modulo condiviso `boss_eval_utils`, titolo parametrico)
9. Efficienza: GMACs / GFLOPs / parametri su grafo ONNX -> `efficiency/flops_params.csv`

---
**Output - `/kaggle/working/TEST_OUTPUT/`:** `model_output/` `gt_eval/` `comparison/` `efficiency/` `markdown/`

---
**Kaggle - dati di input (placeholder da impostare in Cella 1 e Cella 2):**
- **boss_eval_utils.py** (utility-script/dataset) - modulo condiviso di valutazione
- **Model**: PicoDet-S 320 esportato in **ONNX** alla risoluzione di inferenza
- **Dataset** recordings: cartella `images`
- **Dataset** GT: export Roboflow **COCO** (`test/_annotations.coco.json`) - stesso GT di EfficientDet

In [ ]:
# ============================================================
# Cella 1 - Import librerie
# ============================================================
%pip install onnxruntime onnx onnx-tool torchmetrics --quiet

import os
import sys
import shutil
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
from torchmetrics.detection import MeanAveragePrecision
import onnxruntime as ort

import warnings
warnings.filterwarnings('ignore')

In [ ]:
%%writefile boss_eval_utils.py
"""
boss_eval_utils.py — modulo condiviso di valutazione B.O.S.S. OR4

Logica comune ai notebook di test (YOLO11, EfficientDet, NanoDet-Plus, PicoDet)
estratta in un unico punto per garantire confronto a piu modelli omogeneo per
costruzione ed evitare drift tra notebook:

  - mappa classi universale BOSS <-> alias <-> COCO 80
  - soglie identiche (CONF_THRESHOLD, IOU_THRESHOLD, MATCH_IOU)
  - preprocessing uniforme (letterbox, aspect-ratio preservato + padding)
  - greedy matching TP/FP/FN per classe a IoU = MATCH_IOU
  - generazione report markdown con titolo PARAMETRICO sul nome modello reale
  - contatore uniforme GFLOPs/GMACs/parametri su grafo ONNX

Nessuna dipendenza da variabili globali dei notebook: import puro.
"""

import re
import numpy as np
import pandas as pd

try:
    import cv2
except Exception:  # cv2 non necessario per le funzioni di sola valutazione/report
    cv2 = None


# ============================================================
# Soglie identiche per tutti i notebook (fix metodologico C)
# ============================================================
CONF_THRESHOLD = 0.25   # soglia confidenza inferenza/predizioni
IOU_THRESHOLD  = 0.45   # soglia IoU per NMS
MATCH_IOU      = 0.50   # soglia IoU per il matching TP/FP/FN greedy


# ============================================================
# Classi BOSS canoniche + alias
# ============================================================
BOSS_CLASSES = [
    "Bench", "Bicycle Rack", "Bike", "Car", "Chair", "Dustbin",
    "Electrical Box", "Electrical Pole", "Manhole", "Motorcycle",
    "Pedestrian crosswalk", "Person", "Plant Pot", "Road", "Stairs",
    "Table", "Teraffic Barrel", "Traffic sign", "Tree", "Truck",
]
NUM_CLASSES = len(BOSS_CLASSES)

BOSS_ALIASES = {
    "Bench":                ["bench"],
    "Bicycle Rack":         ["bicycle rack", "bike rack", "cycle rack"],
    "Bike":                 ["bike", "bicycle", "cycle"],
    "Car":                  ["car", "automobile"],
    "Chair":                ["chair"],
    "Dustbin":              ["dustbin", "bin", "trash can", "trashcan", "garbage can", "waste bin", "trash"],
    "Electrical Box":       ["electrical box", "electric box", "junction box", "utility box"],
    "Electrical Pole":      ["electrical pole", "electric pole", "utility pole", "power pole", "pole"],
    "Manhole":              ["manhole", "manhole cover"],
    "Motorcycle":           ["motorcycle", "motorbike", "motor bike"],
    "Pedestrian crosswalk": ["pedestrian crosswalk", "crosswalk", "cross walk", "zebra crossing", "pedestrian crossing"],
    "Person":               ["person", "pedestrian", "people", "human"],
    "Plant Pot":            ["plant pot", "potted plant", "pot plant", "flower pot", "flowerpot", "planter"],
    "Road":                 ["road", "street", "roadway"],
    "Stairs":               ["stairs", "staircase", "steps", "stair"],
    "Table":                ["table", "dining table", "desk"],
    "Teraffic Barrel":      ["teraffic barrel", "traffic barrel", "barrel", "traffic drum", "construction barrel"],
    "Traffic sign":         ["traffic sign", "road sign", "street sign", "stop sign", "traffic signal"],
    "Tree":                 ["tree"],
    "Truck":                ["truck", "lorry"],
}


def normalize_name(name):
    return re.sub(r"[\s_\-]+", " ", str(name).strip().lower())


ALIAS_TO_BOSS = {}
for _boss in BOSS_CLASSES:
    ALIAS_TO_BOSS[normalize_name(_boss)] = _boss
    for _alias in BOSS_ALIASES.get(_boss, []):
        ALIAS_TO_BOSS[normalize_name(_alias)] = _boss


def resolve_to_boss(name):
    """Nome (qualsiasi alias/grafia) -> classe BOSS canonica, o None."""
    return ALIAS_TO_BOSS.get(normalize_name(name))


def build_model_to_boss(model_names):
    """{model_id: nome} -> {model_id: boss_index} per le sole classi mappabili."""
    mapping = {}
    for mid, mname in model_names.items():
        boss = resolve_to_boss(mname)
        if boss is not None:
            mapping[int(mid)] = BOSS_CLASSES.index(boss)
    return mapping


# ============================================================
# Spazio classi COCO 80 (NanoDet-Plus, PicoDet, EfficientDet)
# ============================================================
# COCO id 1-based con gap (mancano 12, 26, 29, 30, ...). COCO_ID_TO_SEQ converte
# coco_id 1-based -> indice 0-based sequenziale 0..79: e' l'ordine standard a 80
# classi usato da NanoDet-Plus e PicoDet (class_id del modello = seq_id diretto).
_COCO_RAW = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane",
    6: "bus", 7: "train", 8: "truck", 9: "boat", 10: "traffic light",
    11: "fire hydrant", 13: "stop sign", 14: "parking meter", 15: "bench",
    16: "bird", 17: "cat", 18: "dog", 19: "horse", 20: "sheep",
    21: "cow", 22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe",
    27: "backpack", 28: "umbrella", 31: "handbag", 32: "tie", 33: "suitcase",
    34: "frisbee", 35: "skis", 36: "snowboard", 37: "sports ball", 38: "kite",
    39: "baseball bat", 40: "baseball glove", 41: "skateboard", 42: "surfboard",
    43: "tennis racket", 44: "bottle", 46: "wine glass", 47: "cup",
    48: "fork", 49: "knife", 50: "spoon", 51: "bowl", 52: "banana",
    53: "apple", 54: "sandwich", 55: "orange", 56: "broccoli", 57: "carrot",
    58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair",
    63: "couch", 64: "potted plant", 65: "bed", 67: "dining table",
    70: "toilet", 72: "tv", 73: "laptop", 74: "mouse", 75: "remote",
    76: "keyboard", 77: "cell phone", 78: "microwave", 79: "oven",
    80: "toaster", 81: "sink", 82: "refrigerator", 84: "book",
    85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear",
    89: "hair drier", 90: "toothbrush",
}

_coco_keys_sorted = sorted(_COCO_RAW.keys())
COCO_ID_TO_SEQ = {cid: idx for idx, cid in enumerate(_coco_keys_sorted)}  # 1-based -> 0-based
SEQ_TO_COCO_ID = {v: k for k, v in COCO_ID_TO_SEQ.items()}                # 0-based -> 1-based


def build_coco_model_classes():
    """Ritorna (MODEL_CLASSES, MODEL_NC) per i detector COCO 80 seq."""
    model_classes = {COCO_ID_TO_SEQ[k]: v for k, v in _COCO_RAW.items()}
    return model_classes, len(model_classes)


# ============================================================
# Preprocessing uniforme: letterbox (fix metodologico B)
# ============================================================
def letterbox(img_bgr, new_shape, color=(114, 114, 114)):
    """
    Resize con aspect-ratio preservato + padding (no stretch-resize).

    Ritorna (padded, ratio, (pad_left, pad_top)):
      - padded: immagine new_shape (H, W)
      - ratio:  fattore di scala applicato all'immagine originale
      - pad:    offset (left, top) in pixel del contenuto dentro padded

    Inversa per riportare una box dallo spazio rete a quello originale:
      x_orig = (x_net - pad_left) / ratio
      y_orig = (y_net - pad_top)  / ratio
    """
    if cv2 is None:
        raise RuntimeError("OpenCV (cv2) non disponibile: letterbox richiede cv2.")
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)
    h0, w0 = img_bgr.shape[:2]
    r = min(new_shape[0] / h0, new_shape[1] / w0)
    new_unpad = (int(round(w0 * r)), int(round(h0 * r)))  # (w, h)
    dw = (new_shape[1] - new_unpad[0]) / 2.0
    dh = (new_shape[0] - new_unpad[1]) / 2.0
    resized = cv2.resize(img_bgr, new_unpad, interpolation=cv2.INTER_LINEAR)
    top    = int(round(dh - 0.1))
    bottom = int(round(dh + 0.1))
    left   = int(round(dw - 0.1))
    right  = int(round(dw + 0.1))
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right,
                                cv2.BORDER_CONSTANT, value=color)
    return padded, r, (left, top)


def scale_boxes_from_letterbox(boxes_xyxy, ratio, pad, orig_hw):
    """
    Riporta box [N,4] xyxy dallo spazio rete (letterbox) all'immagine originale,
    con clipping ai bordi.
    """
    if len(boxes_xyxy) == 0:
        return boxes_xyxy
    left, top = pad
    h0, w0 = orig_hw
    out = boxes_xyxy.copy().astype(np.float64)
    out[:, [0, 2]] = (out[:, [0, 2]] - left) / ratio
    out[:, [1, 3]] = (out[:, [1, 3]] - top) / ratio
    out[:, [0, 2]] = np.clip(out[:, [0, 2]], 0, w0)
    out[:, [1, 3]] = np.clip(out[:, [1, 3]], 0, h0)
    return out


# ============================================================
# IoU + greedy matching TP/FP/FN (fix metodologico C)
# ============================================================
def iou_one_to_many(box, boxes):
    """IoU tra una box [x1,y1,x2,y2] e un array [M,4]."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros((0,), dtype=np.float64)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    iw = np.clip(x2 - x1, 0, None)
    ih = np.clip(y2 - y1, 0, None)
    inter = iw * ih
    area_b = (box[2] - box[0]) * (box[3] - box[1])
    area_o = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area_b + area_o - inter + 1e-9)


def accumulate_matches(tp_pc, fp_pc, fn_pc,
                       pred_boxes, pred_labels, pred_scores,
                       gt_boxes, gt_labels, match_iou=MATCH_IOU):
    """
    Greedy matching per classe (predizioni per confidenza decrescente) a IoU
    >= match_iou. Aggiorna in place gli accumulatori per classe tp_pc/fp_pc/fn_pc
    (array indicizzati per class_id). Identico in tutti i notebook.
    """
    pred_boxes  = np.asarray(pred_boxes,  dtype=np.float64).reshape(-1, 4)
    pred_labels = np.asarray(pred_labels, dtype=np.int64).reshape(-1)
    pred_scores = np.asarray(pred_scores, dtype=np.float64).reshape(-1)
    gt_boxes    = np.asarray(gt_boxes,    dtype=np.float64).reshape(-1, 4)
    gt_labels   = np.asarray(gt_labels,   dtype=np.int64).reshape(-1)

    if len(pred_labels) == 0 and len(gt_labels) == 0:
        return
    all_cls = np.unique(np.concatenate([pred_labels, gt_labels]))
    for cls in all_cls:
        cls = int(cls)
        pb = pred_boxes[pred_labels == cls]
        ps = pred_scores[pred_labels == cls]
        gb = gt_boxes[gt_labels == cls]
        matched = np.zeros(len(gb), dtype=bool)
        for pi in np.argsort(-ps):
            ious = iou_one_to_many(pb[pi], gb)
            if len(ious):
                ious = ious.copy()
                ious[matched] = -1.0
                best = int(ious.argmax())
                if ious[best] >= match_iou:
                    matched[best] = True
                    tp_pc[cls] += 1
                    continue
            fp_pc[cls] += 1
        fn_pc[cls] += int((~matched).sum())


def precision_recall_per_class(tp_pc, fp_pc, fn_pc):
    """Da accumulatori TP/FP/FN -> (precision_pc, recall_pc) per classe."""
    tp_pc = np.asarray(tp_pc, dtype=np.float64)
    fp_pc = np.asarray(fp_pc, dtype=np.float64)
    fn_pc = np.asarray(fn_pc, dtype=np.float64)
    denom_p = tp_pc + fp_pc
    denom_r = tp_pc + fn_pc
    prec = np.divide(tp_pc, denom_p, out=np.zeros_like(denom_p), where=denom_p > 0)
    rec  = np.divide(tp_pc, denom_r, out=np.zeros_like(denom_r), where=denom_r > 0)
    return prec, rec


# ============================================================
# Report markdown — titolo PARAMETRICO, nota metodologica corretta
# ============================================================
def df_to_md(df):
    """DataFrame -> tabella Markdown, senza dipendenze esterne."""
    cols = list(df.columns)
    head = "| " + " | ".join(str(c) for c in cols) + " |"
    sep  = "| " + " | ".join("---" for _ in cols) + " |"
    rows = []
    for _, r in df.iterrows():
        cells = [f"{r[c]:.4f}" if isinstance(r[c], float) else str(r[c]) for c in cols]
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join([head, sep] + rows)


def build_report(model_name, config, metrics, df_metrics,
                 class_distribution=None, pred_total=0, frames_with_pred=0,
                 n_frames=0):
    """
    Costruisce il report markdown.

    - model_name: nome modello REALE (titolo parametrico — niente hardcode).
    - config: dict {label: valore} per la sezione Configurazione.
    - metrics: dict con chiavi map50, map5095, precision, recall, f1, match_iou,
               conf_threshold.
    - df_metrics: DataFrame metriche per classe.
    - class_distribution: pd.Series {classe BOSS: conteggio} oppure None.

    Nota metodologica corretta: Precision/Recall reali via TP/FP/FN greedy
    matching (non proxy AP/MAR di torchmetrics).
    """
    cfg_md = "\n".join(f"| {k} | {v} |" for k, v in config.items())

    if class_distribution is not None and len(class_distribution) > 0:
        dist_df = pd.DataFrame({
            "Classe": list(class_distribution.index),
            "Rilevamenti": list(class_distribution.values),
        })
        dist_md = df_to_md(dist_df)
    else:
        dist_md = "_Nessuna predizione._"

    note = (
        f"_Precision e Recall calcolate via greedy matching IoU (TP/FP/FN) a "
        f"IoU = {metrics['match_iou']} e soglia confidenza = {metrics['conf_threshold']}, "
        f"mediate sulle classi BOSS coperte dal modello._"
    )

    report = f"""# B.O.S.S. — Report Test {model_name}

## 1. Configurazione
| Parametro | Valore |
| --- | --- |
{cfg_md}

## 2. Metriche aggregate vs Ground Truth
{note}
| Metrica | Valore |
| --- | --- |
| mAP@0.5 | {metrics['map50']:.4f} |
| mAP@0.5:0.95 | {metrics['map5095']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| F1 Score | {metrics['f1']:.4f} |

## 3. Metriche per classe
{df_to_md(df_metrics)}

## 4. Distribuzione predizioni sui recordings
- Predizioni totali: {pred_total}
- Frame con almeno una predizione: {frames_with_pred} / {n_frames}

{dist_md}
"""
    return report


# ============================================================
# Contatore uniforme GFLOPs/GMACs/parametri su grafo ONNX (fix D)
# ============================================================
def count_onnx_flops_params(onnx_path, input_shape=None, input_name=None,
                            csv_tmp_path=None):
    """
    Conta MACs/parametri su grafo ONNX con onnx-tool (contatore uniforme per
    tutti i notebook). Ritorna dict {params, gmacs, gflops}.

    GMACs = MACs / 1e9 ; GFLOPs = 2 * GMACs (1 MAC = 2 FLOP).

    - onnx_path: percorso del modello ONNX esportato alla risoluzione di inferenza.
    - input_shape / input_name: opzionali; necessari solo se l'ONNX ha shape di
      input dinamiche. Se l'export e' a shape fissa, lasciarli None.
    - csv_tmp_path: file CSV temporaneo per il profilo per-nodo (default accanto
      all'onnx).
    """
    import onnx_tool
    from pathlib import Path

    onnx_path = str(onnx_path)
    if csv_tmp_path is None:
        csv_tmp_path = str(Path(onnx_path).with_suffix(".profile.csv"))

    inputs = None
    if input_shape is not None and input_name is not None:
        from onnx_tool import create_ndarray_f32
        inputs = {input_name: create_ndarray_f32(tuple(input_shape))}

    # model_profile stampa il profilo e (savenode) salva il dettaglio per-nodo.
    onnx_tool.model_profile(onnx_path, inputs, savenode=csv_tmp_path)

    df = pd.read_csv(csv_tmp_path)
    cols = {c.strip().lower(): c for c in df.columns}
    mac_col = cols.get("macs") or cols.get("forwardmacs")
    par_col = cols.get("params") or cols.get("parameters") or cols.get("weight")
    name_col = df.columns[0]
    if mac_col is None or par_col is None:
        raise RuntimeError(f"Colonne MACs/Params non trovate nel CSV onnx-tool: {list(df.columns)}")

    def _num(x):
        s = re.sub(r"[^0-9]", "", str(x))
        return int(s) if s else 0

    # Esclude eventuale riga 'Total' per non raddoppiare: si somma il per-nodo.
    mask = df[name_col].astype(str).str.strip().str.lower() != "total"
    total_macs   = int(df.loc[mask, mac_col].map(_num).sum())
    total_params = int(df.loc[mask, par_col].map(_num).sum())

    gmacs = total_macs / 1e9
    return {"params": total_params, "gmacs": gmacs, "gflops": 2.0 * gmacs}


In [ ]:
# ============================================================
# Cella 1b - Import modulo condiviso (scritto dalla cella precedente)
# ============================================================
import importlib
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import boss_eval_utils as beu
importlib.reload(beu)   # ricarica se la cella %%writefile e' stata rieseguita
print(f"boss_eval_utils caricato da: {beu.__file__}")

In [ ]:
# ============================================================
# Cella 2 - Configurazione + classi (BOSS / COCO) + parametri decode
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

# Nome modello REALE (titolo report parametrico - niente hardcode)
MODEL_NAME = "PicoDet-S 320"

if IS_KAGGLE:
    # <<< INSERIRE PATH KAGGLE CORRETTI >>>
    MODEL_ONNX     = Path("/kaggle/input/PLACEHOLDER-picodet-onnx/picodet_s_320_coco.onnx")
    RECORDINGS_DIR = Path("/kaggle/input/datasets/lorenzoverdura/boss-recordings/images")
    GT_COCO_DIR    = Path("/kaggle/input/boss-gt-coco")
else:
    BASE_DIR       = Path("/home/lorenzoverdura8/BOSS_CODE")
    MODEL_ONNX     = BASE_DIR / "picodet_s_320_coco.onnx"
    RECORDINGS_DIR = BASE_DIR / "recordings"
    GT_COCO_DIR    = BASE_DIR / "gt_coco"

OUTPUT_DIR     = Path("/kaggle/working")
TEST_OUTPUT    = OUTPUT_DIR / "TEST_OUTPUT"
DIR_MODEL_OUT  = TEST_OUTPUT / "model_output"
DIR_GT_EVAL    = TEST_OUTPUT / "gt_eval"
DIR_COMPARISON = TEST_OUTPUT / "comparison"
DIR_EFFICIENCY = TEST_OUTPUT / "efficiency"
DIR_MARKDOWN   = TEST_OUTPUT / "markdown"

# --- Soglie IDENTICHE per tutti i notebook (dal modulo condiviso) ---
CONF_THRESHOLD = beu.CONF_THRESHOLD   # 0.25
IOU_THRESHOLD  = beu.IOU_THRESHOLD    # 0.45 (NMS)
MATCH_IOU      = beu.MATCH_IOU        # 0.50 (TP/FP/FN)

# --- Classi ---
BOSS_CLASSES = beu.BOSS_CLASSES
NUM_CLASSES  = beu.NUM_CLASSES
MODEL_CLASSES, MODEL_NC = beu.build_coco_model_classes()   # COCO 80 seq (class_id = seq_id)
MODEL_TO_BOSS = beu.build_model_to_boss(MODEL_CLASSES)
NUM_MODEL_CLASSES = MODEL_NC
DEVICE = "cpu"
MODEL_PATH = MODEL_ONNX

# ============================================================
# PARAMETRI DECODE PicoDet-S 320 - VERIFICARE SUL CHECKPOINT SCELTO
# ============================================================
# PicoDet-S: input 320 (PicoDet-M: verificare risoluzione del checkpoint, spesso 320 o 416 -> aggiornare INPUT_SIZE e ONNX). Centri cella offset 0.5, normalizzazione RGB ImageNet (scale 1/255, mean/std 0-1), cls gia sigmoid in molti export (se score fuori [0,1] impostare CLS_SIGMOID=True).
INPUT_SIZE  = 320            # risoluzione nativa raccomandata
STRIDES     = (8, 16, 32, 64)     # livelli FPN (GFL)
REG_MAX     = 7                # bin distribuzione box (canali box = 4*(REG_MAX+1))
CELL_OFFSET = 0.5              # offset centro cella (PicoDet)
TO_RGB      = True            # ordine canali atteso dal modello
NORM_SCALE  = 0.00392156862745098             # fattore scala pre-normalizzazione
NORM_MEAN   = np.array([0.485, 0.456, 0.406], dtype=np.float32)
NORM_STD    = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CLS_SIGMOID = False            # True se l'ONNX emette logit cls (applica sigmoid)
SCORE_FLOOR = 0.05            # floor minimo per torchmetrics (analogo post-NMS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if TEST_OUTPUT.exists():
    shutil.rmtree(TEST_OUTPUT)
for d in (DIR_MODEL_OUT, DIR_GT_EVAL, DIR_COMPARISON, DIR_EFFICIENCY, DIR_MARKDOWN):
    d.mkdir(parents=True, exist_ok=True)

mapped_names = sorted({BOSS_CLASSES[b] for b in MODEL_TO_BOSS.values()})
missing      = [c for c in BOSS_CLASSES if c not in mapped_names]
print(f"Ambiente:        {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"MODEL_NAME:      {MODEL_NAME}")
print(f"MODEL_ONNX:      {MODEL_ONNX}  -  esiste: {MODEL_ONNX.exists()}")
print(f"RECORDINGS_DIR:  {RECORDINGS_DIR}  -  esiste: {RECORDINGS_DIR.exists()}")
print(f"GT_COCO_DIR:     {GT_COCO_DIR}  -  esiste: {GT_COCO_DIR.exists()}")
print(f"Classi COCO:     {MODEL_NC}")
print(f"Classi BOSS coperte ({len(mapped_names)}/{NUM_CLASSES}): {mapped_names}")
print(f"Classi BOSS NON coperte: {missing}")
print(f"INPUT_SIZE/STRIDES/REG_MAX: {INPUT_SIZE} / {STRIDES} / {REG_MAX}")
print(f"CONF/IOU/MATCH:  {CONF_THRESHOLD} / {IOU_THRESHOLD} / {MATCH_IOU}")

In [ ]:
# ============================================================
# Cella 3 - Ricerca frame recordings (Kaggle: dataset gia estratto)
# ============================================================
recordings_root = RECORDINGS_DIR
if not recordings_root.exists():
    raise FileNotFoundError(f"Cartella recordings non trovata: {recordings_root}")

TEST_FRAMES_DIR = OUTPUT_DIR / "frames_all"
if TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
TEST_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted(
    p for p in recordings_root.rglob("*")
    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
if not img_files:
    raise FileNotFoundError(
        f"Nessuna immagine (.jpg/.jpeg/.png) trovata sotto {recordings_root}"
    )

all_frames = []
for img in img_files:
    dst = TEST_FRAMES_DIR / img.name
    try:
        dst.symlink_to(img.resolve())
    except OSError:
        shutil.copy(img, dst)
    all_frames.append(dst)

print(f"Totale frame trovati: {len(all_frames)}")
print(f"TEST_FRAMES_DIR (consolidata): {TEST_FRAMES_DIR}")

In [ ]:
# ============================================================
# Cella 4 - ONNX Runtime session + decode GFL + run_detector()
# ============================================================
# NanoDet-Plus e PicoDet condividono lo stesso schema di testa (Generalized
# Focal Loss): per ogni punto griglia -> [num_classes scores] + [4*(reg_max+1)
# logits di distribuzione]. Decode: sigmoid(score) e integrale softmax sulle
# distribuzioni -> distanze (l,t,r,b) -> distance2bbox. NMS per classe.
# I parametri model-specifici (STRIDES, REG_MAX, CELL_OFFSET, normalizzazione,
# CLS_SIGMOID, ...) sono definiti in Cella 2.

session = ort.InferenceSession(str(MODEL_ONNX), providers=["CPUExecutionProvider"])
_inp = session.get_inputs()[0]
ONNX_INPUT_NAME = _inp.name
_ishape = _inp.shape
INPUT_H = int(_ishape[2]) if isinstance(_ishape[2], int) else INPUT_SIZE
INPUT_W = int(_ishape[3]) if isinstance(_ishape[3], int) else INPUT_SIZE
REG_BINS = 4 * (REG_MAX + 1)

print(f"Input ONNX: name='{ONNX_INPUT_NAME}'  shape={_ishape}  -> {INPUT_H}x{INPUT_W}")
print("Output ONNX:")
for o in session.get_outputs():
    print(f"  name='{o.name}'  shape={o.shape}")


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def _make_centers(input_h, input_w, strides, cell_offset):
    """Centri (cx,cy) e stride per ogni punto, ordine livelli = strides,
    intra-livello x veloce / y lento (meshgrid 'ij' flatten row-major)."""
    centers, strs = [], []
    for s in strides:
        fh = int(math.ceil(input_h / s))
        fw = int(math.ceil(input_w / s))
        ys, xs = np.meshgrid(np.arange(fh), np.arange(fw), indexing="ij")
        cx = (xs.reshape(-1) + cell_offset) * s
        cy = (ys.reshape(-1) + cell_offset) * s
        centers.append(np.stack([cx, cy], axis=1))
        strs.append(np.full(fh * fw, s, dtype=np.float64))
    return np.concatenate(centers, 0).astype(np.float64), np.concatenate(strs, 0)


CENTERS, STRS = _make_centers(INPUT_H, INPUT_W, STRIDES, CELL_OFFSET)
print(f"Punti griglia attesi (strides {STRIDES}): {len(CENTERS)}")


def _assemble_gfl(outs):
    """Da output ONNX -> (cls_scores [N,C], reg [N,REG_BINS]).
    Gestisce: output singolo concatenato, due output (cls / reg), oppure
    output per-livello (concatenati in ordine di stride)."""
    cls_parts, reg_parts, single = [], [], None
    for o in outs:
        a = np.asarray(o)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        if a.ndim != 2:
            continue
        if a.shape[1] == NUM_MODEL_CLASSES:
            cls_parts.append(a)
        elif a.shape[1] == REG_BINS:
            reg_parts.append(a)
        elif a.shape[1] == NUM_MODEL_CLASSES + REG_BINS:
            single = a
    if single is not None:
        return single[:, :NUM_MODEL_CLASSES], single[:, NUM_MODEL_CLASSES:]
    if cls_parts and reg_parts:
        return np.concatenate(cls_parts, 0), np.concatenate(reg_parts, 0)
    raise RuntimeError(
        "Output ONNX non riconosciuto. Shapes=" + str([np.asarray(o).shape for o in outs])
    )


def _decode_gfl(cls_scores, reg):
    """GFL decode -> (boxes_xyxy_net [N,4], scores [N,C]) in coordinate rete."""
    if CLS_SIGMOID:
        cls_scores = _sigmoid(cls_scores)
    n = reg.shape[0]
    reg = reg.reshape(n, 4, REG_MAX + 1)
    reg = _softmax(reg, axis=-1)
    proj = np.arange(REG_MAX + 1, dtype=np.float64)
    dist = (reg * proj).sum(-1) * STRS[:, None]   # [N,4] pixel: l,t,r,b
    x1 = CENTERS[:, 0] - dist[:, 0]
    y1 = CENTERS[:, 1] - dist[:, 1]
    x2 = CENTERS[:, 0] + dist[:, 2]
    y2 = CENTERS[:, 1] + dist[:, 3]
    return np.stack([x1, y1, x2, y2], axis=1), cls_scores


def _nms_per_class(boxes, labels, scores, iou_thr):
    final_b, final_l, final_s = [], [], []
    for c in np.unique(labels):
        m = labels == c
        bb, ss = boxes[m], scores[m]
        if len(bb) == 0:
            continue
        rects = [[float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])] for b in bb]
        idxs = cv2.dnn.NMSBoxes(rects, [float(s) for s in ss], 0.0, float(iou_thr))
        if idxs is None or len(idxs) == 0:
            continue
        for i in np.array(idxs).reshape(-1):
            final_b.append(bb[int(i)]); final_l.append(int(c)); final_s.append(float(ss[int(i)]))
    if final_b:
        return (np.array(final_b, dtype=np.float64),
                np.array(final_l, dtype=np.int64),
                np.array(final_s, dtype=np.float64))
    return (np.zeros((0, 4)), np.array([], dtype=np.int64), np.array([]))


def run_detector(img_bgr):
    """Inferenza su immagine BGR -> (boxes_xyxy_abs, seq_ids, scores).
    seq_ids = class_id COCO 0..79 (= seq_id diretto). Preprocessing letterbox."""
    h0, w0 = img_bgr.shape[:2]
    padded, ratio, pad = beu.letterbox(img_bgr, (INPUT_H, INPUT_W))
    img = padded.astype(np.float32)
    if TO_RGB:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img * NORM_SCALE
    img = (img - NORM_MEAN) / NORM_STD
    blob = np.ascontiguousarray(img.transpose(2, 0, 1)[None].astype(np.float32))

    outs = session.run(None, {ONNX_INPUT_NAME: blob})
    cls_scores, reg = _assemble_gfl(outs)
    if cls_scores.shape[0] != len(CENTERS):
        raise RuntimeError(
            f"Mismatch punti: output={cls_scores.shape[0]} vs griglia={len(CENTERS)}. "
            f"Controllare STRIDES/REG_MAX/INPUT_SIZE in Cella 2."
        )

    boxes_net, scores_all = _decode_gfl(cls_scores, reg)
    labels = scores_all.argmax(1).astype(np.int64)
    conf = scores_all.max(1)
    keep = conf >= SCORE_FLOOR
    boxes_net, labels, conf = boxes_net[keep], labels[keep], conf[keep]

    boxes_abs = beu.scale_boxes_from_letterbox(boxes_net, ratio, pad, (h0, w0))
    return _nms_per_class(boxes_abs, labels, conf, IOU_THRESHOLD)


# -- Debug: prima inferenza + range score (logits vs probabilita) -----------
_dbg = next((p for p in TEST_FRAMES_DIR.iterdir()
             if p.suffix.lower() in (".jpg", ".jpeg", ".png")), None)
if _dbg is not None:
    _img = cv2.imread(str(_dbg))
    _outs = session.run(None, {ONNX_INPUT_NAME:
        np.ascontiguousarray((((cv2.cvtColor(beu.letterbox(_img, (INPUT_H, INPUT_W))[0], cv2.COLOR_BGR2RGB)
            if TO_RGB else beu.letterbox(_img, (INPUT_H, INPUT_W))[0]).astype(np.float32) * NORM_SCALE - NORM_MEAN) / NORM_STD)
            .transpose(2, 0, 1)[None].astype(np.float32))})
    _cls, _reg = _assemble_gfl(_outs)
    print(f"\nDebug ({_dbg.name}): cls shape={_cls.shape}  reg shape={_reg.shape}")
    print(f"  cls range raw: [{_cls.min():.3f}, {_cls.max():.3f}]  "
          f"(in [0,1] -> gia sigmoid: CLS_SIGMOID=False; oltre -> logits: CLS_SIGMOID=True)")
    _b, _l, _s = run_detector(_img)
    print(f"  detections post-NMS: {len(_b)}  | seq_ids: {_l[:10]}  | scores: {np.round(_s[:10], 3)}")

In [ ]:
# ============================================================
# Cella 5 - Inferenza su tutti i frame -> df_preds + frame annotati
# ============================================================
all_predictions  = []
PREDICT_SAVE_DIR = DIR_MODEL_OUT / "annotated_frames"
PREDICT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

frame_list = sorted(
    p for p in TEST_FRAMES_DIR.iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)

for img_path in tqdm(frame_list, desc=f"{MODEL_NAME} inference"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    frame_name = img_path.name
    boxes_xyxy, seq_ids, scores = run_detector(img_bgr)
    ann = img_bgr.copy()

    for box, sid, score in zip(boxes_xyxy, seq_ids, scores):
        if score < CONF_THRESHOLD or sid < 0:
            continue
        boss_id    = MODEL_TO_BOSS.get(int(sid))
        class_name = MODEL_CLASSES.get(int(sid), str(sid))
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   int(sid),
            "class_name": class_name,
            "boss_class": BOSS_CLASSES[boss_id] if boss_id is not None else None,
            "confidence": float(score),
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
        })
        cv2.rectangle(ann, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0,255,0), 2)
        cv2.putText(ann, f"{class_name} {score:.2f}",
                    (int(box[0]), max(int(box[1])-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    cv2.imwrite(str(PREDICT_SAVE_DIR / frame_name), ann)

df_preds = pd.DataFrame(all_predictions)
print(f"\nFrame processati:  {len(frame_list)}")
print(f"Predizioni totali: {len(df_preds)}")
print(f"Frame con pred.:   {df_preds['frame'].nunique() if not df_preds.empty else 0}")
print(f"Annotati in:       {PREDICT_SAVE_DIR}")
df_preds.to_csv(DIR_MODEL_OUT / "predictions.csv", index=False)
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Cella 6 - Distribuzione classi (BOSS) e confidenza
# ============================================================
if df_preds.empty:
    print("Nessuna predizione - grafici saltati.")
else:
    df_boss = df_preds.dropna(subset=["boss_class"])
    class_counts = (
        df_boss["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = plt.cm.tab20.colors[:NUM_CLASSES]
    ax.bar(class_counts.index, class_counts.values, color=colors)
    ax.set_title("Distribuzione classi BOSS rilevate - recordings", fontsize=14)
    ax.set_xlabel("Classe BOSS")
    ax.set_ylabel("Numero rilevamenti")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_class_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(CONF_THRESHOLD, color="red", linestyle="--", label=f"soglia={CONF_THRESHOLD}")
    ax.set_title("Distribuzione confidenza predizioni", fontsize=13)
    ax.set_xlabel("Confidenza")
    ax.set_ylabel("Frequenza")
    ax.legend()
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_confidence_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 7 - Ground Truth (COCO JSON Roboflow) + valutazione
# ============================================================
# Stesso GT COCO canonico del notebook EfficientDet (stesse immagini test).
# mAP@0.5, mAP@0.5:0.95 con torchmetrics; Precision/Recall reali via greedy
# matching TP/FP/FN (modulo condiviso) a IoU=MATCH_IOU, conf=CONF_THRESHOLD.

gt_json_path = GT_COCO_DIR / "test" / "_annotations.coco.json"
GT_TEST_IMGS = GT_COCO_DIR / "test"

if not gt_json_path.exists():
    raise FileNotFoundError(f"Annotazioni COCO non trovate: {gt_json_path}")

with open(gt_json_path) as f:
    gt_data = json.load(f)

print(f"GT COCO: {len(gt_data['images'])} immagini, "
      f"{len(gt_data['annotations'])} annotazioni, "
      f"{len(gt_data['categories'])} categorie")

cat_id_to_seq = {}
unmapped_cats = []
for cat in gt_data["categories"]:
    boss = beu.resolve_to_boss(cat["name"])
    seq = None
    if boss is not None:
        for seq_id, coco_name in MODEL_CLASSES.items():
            if beu.resolve_to_boss(coco_name) == boss:
                seq = seq_id
                break
    if seq is None:
        unmapped_cats.append(cat["name"])
    else:
        cat_id_to_seq[cat["id"]] = seq
print(f"Categorie GT mappate (cat_id -> seq_id): {cat_id_to_seq}")
print(f"Categorie GT scartate (non rilevabili dal modello): {unmapped_cats}")

img_id_to_anns = {}
for ann in gt_data["annotations"]:
    img_id_to_anns.setdefault(ann["image_id"], []).append(ann)

metric_50   = MeanAveragePrecision(iou_type="bbox", class_metrics=True, iou_thresholds=[0.5])
metric_full = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

tp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fn_pc = np.zeros(MODEL_NC, dtype=np.int64)

processed = 0
for img_info in tqdm(gt_data["images"], desc=f"Valutazione GT ({MODEL_NAME})"):
    img_path = GT_TEST_IMGS / img_info["file_name"]
    img_bgr  = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    gt_boxes, gt_labels = [], []
    for ann in img_id_to_anns.get(img_info["id"], []):
        sid = cat_id_to_seq.get(ann["category_id"])
        if sid is None:
            continue
        x, y, bw, bh = ann["bbox"]
        gt_boxes.append([x, y, x + bw, y + bh])
        gt_labels.append(sid)
    if not gt_boxes:
        continue

    targets = [{"boxes":  torch.tensor(gt_boxes,  dtype=torch.float32),
                "labels": torch.tensor(gt_labels, dtype=torch.int64)}]

    boxes_xyxy, seq_ids, scores = run_detector(img_bgr)
    valid = seq_ids >= 0
    preds = [{"boxes":  torch.tensor(boxes_xyxy[valid], dtype=torch.float32),
              "scores": torch.tensor(scores[valid],     dtype=torch.float32),
              "labels": torch.tensor(seq_ids[valid],    dtype=torch.int64)}]

    metric_50.update(preds, targets)
    metric_full.update(preds, targets)

    keep = valid & (scores >= CONF_THRESHOLD)
    beu.accumulate_matches(
        tp_pc, fp_pc, fn_pc,
        boxes_xyxy[keep], seq_ids[keep], scores[keep],
        gt_boxes, gt_labels, MATCH_IOU,
    )
    processed += 1

print(f"Immagini valutate: {processed}")

res_50   = metric_50.compute()
res_full = metric_full.compute()

map50   = float(res_50["map"])
map5095 = float(res_full["map"])

classes_t = res_50["classes"]
ap50_t    = res_50["map_per_class"]
ap5095_t  = res_full["map_per_class"]

valid_cls           = (ap50_t >= 0) & (ap5095_t >= 0)
gt_class_indices    = classes_t[valid_cls].numpy().astype(int)
gt_per_class_ap50   = ap50_t[valid_cls].numpy()
gt_per_class_ap5095 = ap5095_t[valid_cls].numpy()

prec_pc, rec_pc = beu.precision_recall_per_class(tp_pc, fp_pc, fn_pc)
gt_per_class_p = prec_pc[gt_class_indices]
gt_per_class_r = rec_pc[gt_class_indices]

precision = float(gt_per_class_p.mean()) if len(gt_per_class_p) > 0 else 0.0
recall    = float(gt_per_class_r.mean()) if len(gt_per_class_r) > 0 else 0.0
f1_score  = 2 * precision * recall / (precision + recall + 1e-9)

GT_EVAL_SAVE_DIR = DIR_GT_EVAL

print(f"\nmAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"Recall:        {recall:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Classi valutate: {len(gt_class_indices)}")

In [ ]:
# ============================================================
# Cella 8 - Metriche per classe + grafici GT
# ============================================================
def _boss_label(model_id):
    bid = MODEL_TO_BOSS.get(int(model_id))
    return BOSS_CLASSES[bid] if bid is not None else MODEL_CLASSES.get(int(model_id), str(model_id))

gt_class_names  = [_boss_label(i) for i in gt_class_indices]
gt_per_class_f1 = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

df_metrics = pd.DataFrame({
    "Classe":      gt_class_names,
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
df_display[float_cols] = df_display[float_cols].applymap(lambda x: f"{x:.4f}")
print(f"=== Metriche per Classe ({MODEL_NAME}) ===")
print(df_display.to_string(index=False))
df_metrics.to_csv(DIR_GT_EVAL / "metrics_per_class_gt.csv", index=False)
print(f"\nSalvato: {DIR_GT_EVAL}/metrics_per_class_gt.csv")

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title(f"AP per classe - Ground Truth ({MODEL_NAME})", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_ap_per_class_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

fig, ax = plt.subplots(figsize=(10, 7))
for _, row in df_plot.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision vs Recall per classe - Ground Truth ({MODEL_NAME})")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_pr_scatter_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 9 - Griglia campione frame annotati
# ============================================================
N_SAMPLES = 9

if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = (
        list(PREDICT_SAVE_DIR.glob("*.jpg")) +
        list(PREDICT_SAVE_DIR.glob("*.png"))
    )

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings - predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_sample_predictions.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 10 - Dashboard riepilogo finale (confronto GT vs modello)
# ============================================================
fig = plt.figure(figsize=(16, 10))
fig.suptitle(f"B.O.S.S. - Riepilogo Test su recordings ({MODEL_NAME})", fontsize=16, fontweight="bold")

ax1 = fig.add_subplot(2, 3, 1)
metrics_summary = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()), color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(2, 3, 2)
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i % len(scatter_colors)], s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(2, 3, 5)
df_boss5 = df_preds.dropna(subset=["boss_class"]) if not df_preds.empty else df_preds
if not df_preds.empty and not df_boss5.empty:
    class_counts_test = df_boss5["boss_class"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 7})
    ax5.set_title("Distribuzione classi BOSS - Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione mappata", ha="center", va="center")
    ax5.axis("off")

ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
frames_count = len(all_frames)
pred_count   = len(df_preds)
pred_mapped  = int(df_preds["boss_class"].notna().sum()) if not df_preds.empty else 0
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
info_text = (
    f"Modello:        {Path(str(MODEL_PATH)).name}\n"
    f"Classi modello: {MODEL_NC}\n"
    f"Classi BOSS:    {NUM_CLASSES}\n"
    f"Input:          {INPUT_H}x{INPUT_W}\n"
    f"Preproc:        letterbox\n"
    f"Conf threshold: {CONF_THRESHOLD}\n"
    f"IoU threshold:  {IOU_THRESHOLD}\n"
    f"Device:         {DEVICE}\n\n"
    f"Frame recordings:  {frames_count}\n"
    f"Pred. totali:      {pred_count}\n"
    f"Pred. mappate BOSS:{pred_mapped}\n"
    f"GT immagini test:  {gt_img_count}\n"
    f"Output:            TEST_OUTPUT/\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
save_path = DIR_COMPARISON / "plot_dashboard_riepilogo.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 11 - Report Markdown (modulo condiviso, titolo parametrico)
# ============================================================
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))

if not df_preds.empty:
    class_dist = (
        df_preds.dropna(subset=["boss_class"])["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    pred_total       = len(df_preds)
    frames_with_pred = df_preds["frame"].nunique()
else:
    class_dist = None
    pred_total = 0
    frames_with_pred = 0

config = {
    "Modello":                     Path(str(MODEL_PATH)).name,
    "Classi modello":              MODEL_NC,
    "Classi BOSS":                 NUM_CLASSES,
    "Input modello":               f"{INPUT_H}x{INPUT_W}",
    "Preprocessing":               "letterbox (aspect-ratio preservato + padding)",
    "Conf threshold (inferenza)":  CONF_THRESHOLD,
    "IoU threshold (NMS)":         IOU_THRESHOLD,
    "Match IoU (TP/FP/FN)":        MATCH_IOU,
    "Device":                      DEVICE,
    "Frame recordings":            len(all_frames),
    "Immagini GT (test)":          gt_img_count,
}
metrics = {
    "map50": map50, "map5095": map5095,
    "precision": precision, "recall": recall, "f1": f1_score,
    "match_iou": MATCH_IOU, "conf_threshold": CONF_THRESHOLD,
}

report = beu.build_report(
    MODEL_NAME, config, metrics, df_metrics,
    class_distribution=class_dist, pred_total=pred_total,
    frames_with_pred=frames_with_pred, n_frames=len(all_frames),
)

report_path = DIR_MARKDOWN / "report.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report Markdown scritto: {report_path}")

metrics_md_path = DIR_MARKDOWN / "metrics_per_class.md"
metrics_md_path.write_text(
    f"# Metriche per classe - Ground Truth ({MODEL_NAME})\n\n" + beu.df_to_md(df_metrics) + "\n",
    encoding="utf-8",
)
print(f"Tabella metriche scritta: {metrics_md_path}")

In [ ]:
# ============================================================
# Cella 12 - Efficienza: GMACs / GFLOPs / parametri su grafo ONNX
# ============================================================
# Contatore uniforme (onnx-tool) per tutti i notebook. Il modello e' gia ONNX
# (lo stesso usato per l'inferenza), alla risoluzione di inferenza effettiva.
# GMACs = MACs/1e9 ; GFLOPs = 2*GMACs. Salvataggio in efficiency/flops_params.csv.
eff = beu.count_onnx_flops_params(
    MODEL_ONNX,
    input_shape=(1, 3, INPUT_H, INPUT_W),
    input_name=ONNX_INPUT_NAME,
)

df_eff = pd.DataFrame([{
    "model":      MODEL_NAME,
    "onnx":       Path(str(MODEL_ONNX)).name,
    "input_h":    INPUT_H,
    "input_w":    INPUT_W,
    "params":     eff["params"],
    "params_M":   round(eff["params"] / 1e6, 4),
    "gmacs":      round(eff["gmacs"], 4),
    "gflops":     round(eff["gflops"], 4),
}])
eff_csv = DIR_EFFICIENCY / "flops_params.csv"
df_eff.to_csv(eff_csv, index=False)
print(df_eff.to_string(index=False))
print(f"\nSalvato: {eff_csv}")